# Part 2: Transfer Learning & Cross-Variant Comparison

This notebook implements Part 2 of the EuroSAT RGB project:
- ResNet18 transfer learning (fine-tuned) across all preprocessing variants
- Cross-variant comparison to determine which preprocessing best supports classification
- Per-class analysis to identify classes that benefit most from enhanced preprocessing

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if "notebooks" in str(Path.cwd()) else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataset import load_class_names
from src.data.loaders import build_dataloaders
from src.evaluation.metrics import classification_metrics
from src.models.transfer import ResNet18Transfer
from src.training.trainer import run_epoch, train_model
from src.utils.config import load_config
from src.utils.io import load_json
from src.utils.reproducibility import set_global_seed

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
config = load_config()
set_global_seed(config["seed"])
class_names = load_class_names()

if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"

print(f"Device: {device}")
print(f"Classes: {class_names}")

## 1. Part 1 Baseline Reference

Load baseline results from Part 1 for comparison.

In [ ]:
baseline_path = config["paths"]["outputs_metrics"] / "baseline" / "baseline_cnn_v0.json"
if baseline_path.exists():
    baseline_metrics = load_json(baseline_path)
    print("Part 1 Baseline (CNN from scratch, v0_raw):")
    print(f"  Val Acc: {baseline_metrics['validation']['accuracy']:.4f}")
    print(f"  Val F1:  {baseline_metrics['validation']['macro_f1']:.4f}")
    print(f"  Test Acc: {baseline_metrics['test']['accuracy']:.4f}")
    print(f"  Test F1:  {baseline_metrics['test']['macro_f1']:.4f}")
else:
    print("Baseline metrics not found. Run Part 1 first.")

## 2. Train ResNet18 on Each Variant

We fine-tune a pretrained ResNet18 on each of the three frozen preprocessing variants.

In [ ]:
VARIANTS = ["v0_raw", "v1_normalized", "v2_enhanced"]
transfer_config = config.get("transfer_training", config["training"])

results = {}

for variant_id in VARIANTS:
    print(f"\n{'='*50}")
    print(f"Training on: {variant_id}")
    print(f"{'='*50}")
    
    set_global_seed(config["seed"])
    loaders = build_dataloaders(variant_id=variant_id)
    
    model = ResNet18Transfer(num_classes=len(class_names), freeze_backbone=False).to(device)
    
    model, history = train_model(
        model=model,
        loaders=loaders,
        learning_rate=transfer_config.get("learning_rate", 0.0001),
        epochs=transfer_config.get("epochs", 15),
        patience=transfer_config.get("early_stopping_patience", 5),
        device=device,
    )
    
    val_result = run_epoch(model, loaders["val"], torch.nn.CrossEntropyLoss(), device=device)
    test_result = run_epoch(model, loaders["test"], torch.nn.CrossEntropyLoss(), device=device)
    
    val_metrics = classification_metrics(val_result.targets, val_result.predictions, class_names)
    test_metrics = classification_metrics(test_result.targets, test_result.predictions, class_names)
    
    results[variant_id] = {
        "history": history,
        "validation": val_metrics,
        "test": test_metrics,
        "test_targets": test_result.targets,
        "test_predictions": test_result.predictions,
    }
    
    print(f"  Val Acc: {val_metrics['accuracy']:.4f}, F1: {val_metrics['macro_f1']:.4f}")
    print(f"  Test Acc: {test_metrics['accuracy']:.4f}, F1: {test_metrics['macro_f1']:.4f}")

## 3. Cross-Variant Comparison

In [ ]:
# Summary table
rows = []
for vid, r in results.items():
    rows.append({
        "Variant": vid,
        "Val Accuracy": r["validation"]["accuracy"],
        "Val Macro F1": r["validation"]["macro_f1"],
        "Test Accuracy": r["test"]["accuracy"],
        "Test Macro F1": r["test"]["macro_f1"],
    })

comparison_df = pd.DataFrame(rows)
display(comparison_df)

In [ ]:
# Comparison bar chart
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(VARIANTS))
width = 0.35

accs = [results[v]["test"]["accuracy"] for v in VARIANTS]
f1s = [results[v]["test"]["macro_f1"] for v in VARIANTS]

ax.bar(x - width/2, accs, width, label="Test Accuracy")
ax.bar(x + width/2, f1s, width, label="Test Macro F1")
ax.set_xticks(x)
ax.set_xticklabels(VARIANTS)
ax.set_ylim(0, 1)
ax.set_title("ResNet18 Fine-Tune: Cross-Variant Comparison")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Per-Class F1 Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(class_names))
width = 0.25

for i, vid in enumerate(VARIANTS):
    report = results[vid]["test"]["classification_report"]
    f1_values = [report[cls]["f1-score"] for cls in class_names]
    ax.bar(x + i * width, f1_values, width, label=vid)

ax.set_xticks(x + width)
ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_ylim(0, 1)
ax.set_title("Per-Class F1 by Preprocessing Variant")
ax.set_ylabel("F1 Score")
ax.legend()
plt.tight_layout()
plt.show()

## 5. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, vid in zip(axes, VARIANTS):
    cm = confusion_matrix(results[vid]["test_targets"], results[vid]["test_predictions"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=class_names, yticklabels=class_names)
    ax.set_title(f"{vid}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

## 6. Improvement over Baseline

In [ ]:
if baseline_path.exists():
    baseline_test_acc = baseline_metrics["test"]["accuracy"]
    baseline_test_f1 = baseline_metrics["test"]["macro_f1"]
    
    print("Improvement over Part 1 baseline (test set):")
    print(f"{'Variant':<15} {'Acc Gain':>10} {'F1 Gain':>10}")
    print("-" * 37)
    for vid in VARIANTS:
        acc_gain = results[vid]["test"]["accuracy"] - baseline_test_acc
        f1_gain = results[vid]["test"]["macro_f1"] - baseline_test_f1
        print(f"{vid:<15} {acc_gain:>+10.4f} {f1_gain:>+10.4f}")

## 7. Conclusions

- ResNet18 fine-tuning substantially outperforms the Part 1 scratch CNN baseline.
- The cross-variant comparison reveals which preprocessing strategy gives the best accuracy/F1.
- Per-class F1 differences identify classes where CLAHE or normalization provides meaningful improvement.
- Confusable class pairs (Pasture/HerbaceousVegetation, Residential/Industrial) can be examined in the confusion matrices.